In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Au fost detectate {len(gpus)} GPU-uri active:")
    for i, gpu in enumerate(gpus):
        print(f" Placa {i}: {gpu.name}")
else:
    print("Niciun GPU detectat. Sistemul rulează lent pe CPU.")

Au fost detectate 2 GPU-uri active:
 Placa 0: /physical_device:GPU:0
 Placa 1: /physical_device:GPU:1


In [13]:
!pip install ai_edge_litert
from ai_edge_litert.interpreter import Interpreter

In [4]:
strategy = tf.distribute.MirroredStrategy()
print(f"Numărul de unități grafice sincronizate: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Numărul de unități grafice sincronizate: 2


In [5]:
from tensorflow import keras

modele = [
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/mobilenet_gpu_fer-2013-augmented.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_ck.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_fer-2013-original.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vrescnn_gpu_raf-db.keras"
]

for path in modele:
    model = keras.models.load_model(path)
    print(f"{path.split('/')[-1]} → input shape: {model.input_shape}")

mobilenet_gpu_fer-2013-augmented.keras → input shape: (None, 48, 48, 1)
vcnn_gpu_ck.keras → input shape: (None, 48, 48, 1)
vcnn_gpu_fer-2013-original.keras → input shape: (None, 48, 48, 1)
vrescnn_gpu_raf-db.keras → input shape: (None, 48, 48, 1)


In [28]:
from tensorflow import keras
from tensorflow.keras.models import load_model
import tensorflow as tf

model = load_model('/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_fer-2013-original.keras')   
model.export('some', format="tf_saved_model")

mod=tf.saved_model.load('some')

INFO:tensorflow:Assets written to: some/assets


INFO:tensorflow:Assets written to: some/assets


Saved artifact at 'some'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 48, 48, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  135476178323792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178326288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178326096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178323600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178325136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178327248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178327056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178325328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178325904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178327440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476178328784: TensorSpec(s

In [29]:
converter = tf.lite.TFLiteConverter.from_keras_model(mod)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_file='vcnn_gpu_fer-2013-original.tflite'
with open(tflite_file, 'wb') as f:
    f.write(tflite_model)

print("Conversia a fost realizată cu succes și modelul a fost salvat ca: ",tflite_file)

INFO:tensorflow:Assets written to: /tmp/tmpqk1s7aqp/assets


INFO:tensorflow:Assets written to: /tmp/tmpqk1s7aqp/assets


Conversia a fost realizată cu succes și modelul a fost salvat ca:  vcnn_gpu_fer-2013-original.tflite


W0000 00:00:1778164374.203110      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778164374.203147      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


In [31]:
interpreter = Interpreter(model_path="vcnn_gpu_fer-2013-original.tflite") 
interpreter.allocate_tensors()

In [32]:
# celula 1 realizeaza preprocesarea datelor + augumentare
import keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir= '/kaggle/input/datasets/msambare/fer2013/train'
test_dir='/kaggle/input/datasets/msambare/fer2013/test'

print ("Versiune TensorFlow: ", tf.__version__)
print ("Versiune Keras: ", keras.__version__)

img_size=48
batch_size=64
epoci = 50
num_classes= 7 
print ("\n partea de preprocesare si impartire a datelor")
datagen = ImageDataGenerator (
    featurewise_center=False,
    featurewise_std_normalization=False,
    rescale=1./255,
    rotation_range=10,
    zoom_range= 0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8,1.2],
    fill_mode='nearest',
    validation_split=0.20
)

test_datagen = ImageDataGenerator(rescale=1./255)

print("Setul de antrenare (80%):")
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",      
    batch_size=batch_size,
    class_mode='categorical',     
    shuffle=True                  
)

print("Setul de validare (20%):")
val_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,                
    subset='validation'           
)

print("Setul de testare:")
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Versiune TensorFlow:  2.19.0
Versiune Keras:  3.10.0

 partea de preprocesare si impartire a datelor
Setul de antrenare (80%):
Found 28709 images belonging to 7 classes.
Setul de validare (20%):
Found 5741 images belonging to 7 classes.
Setul de testare:
Found 7178 images belonging to 7 classes.


In [33]:
import numpy as np
import time
from sklearn.metrics import classification_report, confusion_matrix

input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

test_generator.reset()
y_true = test_generator.classes
predictions = []
durata_totala = 0

for img_batch, _ in test_generator:
    for img in img_batch:
        img_exp = np.expand_dims(img, axis=0).astype(np.float32)
        interpreter.set_tensor(input_index, img_exp)
        t1 = time.time()
        interpreter.invoke()
        t2 = time.time()
        durata_totala += (t2 - t1)
        output = interpreter.get_tensor(output_index)
        predictions.append(np.argmax(output))
    if len(predictions) >= len(y_true):
        break

predictions = predictions[:len(y_true)]
acc = sum(p == t for p, t in zip(predictions, y_true)) / len(y_true)
latenta = 1000 * durata_totala / len(y_true)

print(f" Acuratețe VCNN FER-2013 original (.tflite): {acc*100:.2f}%")
print(f" Latența per imagine: {latenta:.4f} ms")
cm = confusion_matrix(y_true, predictions)
print(cm)
etichete = list(test_generator.class_indices.keys())
print(classification_report(y_true, predictions, target_names=etichete))

 Acuratețe VCNN FER-2013 original (.tflite): 60.03%
 Latența per imagine: 4.6724 ms
[[ 560    0   44   49  113  152   40]
 [  89    0    8    1    3   10    0]
 [ 218    0  208   27  117  286  168]
 [  55    0   27 1511   86   44   51]
 [  91    0   42   77  770  222   31]
 [ 216    0   82   76  263  585   25]
 [  25    0   61   31   24   15  675]]
              precision    recall  f1-score   support

       angry       0.45      0.58      0.51       958
     disgust       0.00      0.00      0.00       111
        fear       0.44      0.20      0.28      1024
       happy       0.85      0.85      0.85      1774
     neutral       0.56      0.62      0.59      1233
         sad       0.45      0.47      0.46      1247
    surprise       0.68      0.81      0.74       831

    accuracy                           0.60      7178
   macro avg       0.49      0.51      0.49      7178
weighted avg       0.59      0.60      0.58      7178



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [36]:
import os

size_in_bytes = os.path.getsize('vcnn_gpu_fer-2013-original.tflite')
lite_sz = round(size_in_bytes/1000, 2)

print('Raport final VCNN FER-2013 original:', 
      'Lite_sz', str(lite_sz)+'k',
      'acc:', str(round(100*acc, 2)),
      'lat', str(round(latenta, 2)))

Raport final VCNN FER-2013 original: Lite_sz 1610.25k acc: 60.03 lat 4.67
